# Sinmate: QLoRA Fine-Tuning Gemma-3-1B-IT for Spoken Sinhala (Jupyter / Global)

This notebook fine-tunes **`google/gemma-3-1b-it`** using **4-bit QLoRA** on a conversational Sinhala dataset (*Kathitha Bhashawa* with Sri Lankan fillers, Singlish, and 4 personas).

It is the **Jupyter-server twin** of `notebooks/colab/02_colab_qlora_finetune_gemma3.ipynb`: same cell numbers and order, but the Colab-only parts (Colab secrets, `!` shell magics, `/content` paths, the apt-based Ollama install) are replaced with portable code that runs on a regular Jupyter server (local workstation, remote GPU box, JupyterHub). Keep the two notebooks in sync.

### Requirements:
- **NVIDIA GPU + CUDA build of PyTorch**: 4-bit QLoRA (`bitsandbytes`) does not run on CPU-only machines. The free Colab T4 (15GB) is known to work.
- `git`, and internet access to GitHub: Cell 9 downloads llama.cpp's prebuilt `llama-quantize`. Only if that is unavailable does it build from source, which needs `cmake` + a C++ compiler.
- Optional: Ollama for Cells 11-12. On Linux Cell 11 installs it for you under `~/.sinmate/ollama` (no root needed, ~1.5GB download); on macOS/Windows install the [desktop app](https://ollama.com/download) first. Without Ollama, Cell 12 falls back to HuggingFace Transformers.
- A Hugging Face token with Gemma access (`HF_TOKEN` env var, cached `huggingface-cli login`, or paste when prompted).

### Pipeline Overview:
1. **Load Base Model**: `google/gemma-3-1b-it` quantized to 4-bit (VRAM footprint ~1.8GB).
2. **Chat Template**: Standard Gemma turn tags `<start_of_turn>user...<end_of_turn>`.
3. **Train with QLoRA**: LoRA adapters on linear attention and MLP layers (`r=16, alpha=32`).
4. **Inference Test**: Multi-persona conversational test in colloquial Sinhala.
5. **Export GGUF**: Export 4-bit quantized GGUF weights (~700MB–800MB) for offline laptop execution.

In [14]:
# 1. Install fine-tuning dependencies into this kernel's environment
import subprocess
import sys
from importlib import metadata

# Packages this kernel already imported keep their OLD code in memory after the install below.
ALREADY_IMPORTED = sorted(m for m in ("transformers", "peft", "trl", "bitsandbytes", "accelerate") if m in sys.modules)

%pip install -q transformers datasets accelerate peft trl bitsandbytes sentencepiece

# torchao is not needed here (QLoRA uses bitsandbytes), but transformers imports it eagerly whenever it is installed, and
# PEFT rejects versions < 0.16. A torchao built for a newer torch than this environment has (e.g. 0.18 needs torch >= 2.11)
# crashes on import, which then surfaces in Cell 4 as "ImportError: cannot import name 'BloomPreTrainedModel'".
# So keep an existing torchao only if it is new enough AND actually imports; otherwise remove it.
try:
    torchao_version = metadata.version("torchao")
except metadata.PackageNotFoundError:
    torchao_version = None

if torchao_version:
    from packaging.version import Version
    imports_ok = subprocess.run([sys.executable, "-c", "import torchao"], capture_output=True).returncode == 0
    if Version(torchao_version) < Version("0.16.0") or not imports_ok:
        print(f"Removing torchao {torchao_version} (too old for PEFT or incompatible with this torch; not needed for QLoRA)")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)

# 4-bit QLoRA needs an NVIDIA CUDA build of PyTorch (https://pytorch.org/get-started/locally/)
import torch
print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if ALREADY_IMPORTED:
    print(f"⚠️ {', '.join(ALREADY_IMPORTED)} were imported before this install. Restart the kernel (Kernel > Restart) and run from Cell 1 again before Cell 4.")

Note: you may need to restart the kernel to use updated packages.
torch 2.6.0+cu124 | CUDA available: True
⚠️ accelerate, bitsandbytes, peft, transformers, trl were imported before this install. Restart the kernel (Kernel > Restart) and run from Cell 1 again before Cell 4.


In [15]:
# 2. Hugging Face Login (Access Gemma license)
# Token lookup order: HF_TOKEN env var -> cached `huggingface-cli login` -> getpass prompt
import os
from getpass import getpass
from huggingface_hub import get_token, login

token = os.environ.get('HF_TOKEN') or get_token() or getpass('Enter your Hugging Face access token: ')
login(token=token)

In [16]:
# 3. Force a fresh clone of Sinmate and load the dataset

import os
import shutil
import subprocess
from pathlib import Path
from datasets import load_dataset

REPO_URL = "https://github.com/chathuwa-whiz/Sinmate.git"
FORCE_RECLONE = True

# Find the workspace, even if the current directory is inside Sinmate
cwd = Path.cwd().resolve()
existing_repo = next(
    (p for p in [cwd, *cwd.parents] if p.name == "Sinmate"),
    None
)

workspace = existing_repo.parent if existing_repo else cwd
os.chdir(workspace)

repo_root = workspace / "Sinmate"

# Remove the existing folder if a fresh clone is requested
if FORCE_RECLONE and repo_root.exists():
    print(f"Removing existing repository: {repo_root}")
    shutil.rmtree(repo_root)

# Clone the repository
if not repo_root.exists():
    print("Cloning Sinmate from GitHub...")
    subprocess.run(
        ["git", "clone", REPO_URL, str(repo_root)],
        check=True
    )

# Change to the repository directory
os.chdir(repo_root)
print("Working directory:", repo_root)

# Verify that the dataset files exist
train_file = repo_root / "data" / "train.jsonl"
val_file = repo_root / "data" / "val.jsonl"

if not train_file.exists() or not val_file.exists():
    raise FileNotFoundError(
        "Dataset files are missing. Expected data/train.jsonl "
        "and data/val.jsonl inside the repository."
    )

# Load the dataset
dataset = load_dataset(
    "json",
    data_files={
        "train": str(train_file),
        "val": str(val_file),
    }
)

print("Dataset loaded successfully:")
print(dataset)

Removing existing repository: /home/jovyan/Sinmate
Cloning Sinmate from GitHub...


Cloning into '/home/jovyan/Sinmate'...


Working directory: /home/jovyan/Sinmate


Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Dataset loaded successfully:
DatasetDict({
    train: Dataset({
        features: ['id', 'persona', 'input_type', 'emotion', 'scenario', 'messages', 'category'],
        num_rows: 106
    })
    val: Dataset({
        features: ['id', 'persona', 'input_type', 'emotion', 'scenario', 'messages', 'category'],
        num_rows: 11
    })
})


In [17]:
# 4. Load Gemma-3-1B-IT in 4-bit
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "google/gemma-3-1b-it"

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected: 4-bit QLoRA (bitsandbytes) needs an NVIDIA GPU. "
        "Use a CUDA-enabled Jupyter server, or the Colab notebook in notebooks/colab/."
    )

# bf16 needs compute capability >= 8.0 (A100/L4/...). The free-tier T4 is 7.5: bf16 kernels fail there,
# and Gemma 3 overflows in fp16, so fall back to fp32 compute (fine for a 1B model).
USE_BF16 = torch.cuda.get_device_capability()[0] >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float32
print(f"GPU: {torch.cuda.get_device_name()} | compute dtype: {COMPUTE_DTYPE}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=COMPUTE_DTYPE
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

GPU: NVIDIA A40 | compute dtype: torch.bfloat16
trainable params: 13,045,760 || all params: 1,012,931,712 || trainable%: 1.2879


In [18]:
# 5. Format Chat Templates & Train with Response-Only Loss Masking
import math
import re
from trl import SFTConfig, SFTTrainer

# Ensure tokenizer padding token is defined for batched training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Use conversational messages format directly (TRL applies chat template automatically)
train_dataset = dataset["train"].select_columns(["messages"])
val_dataset = dataset["val"].select_columns(["messages"])

# Response-only loss needs a chat template with {% generation %} markers around the assistant turns. Gemma 3's stock
# template has none: older TRL (e.g. 0.23) then fails with "chat template doesn't generate assistant masks", while newer
# TRL patches it itself. This template renders exactly the same text as Gemma 3's, plus the markers. It is only active
# while the trainer tokenizes the dataset; the original is restored afterwards so Cells 7-8 save an unmodified tokenizer.
GEMMA3_TRAINING_TEMPLATE = """{{ bos_token }}
{%- if messages[0]['role'] == 'system' -%}
    {%- set first_user_prefix = messages[0]['content'] + '\n\n' -%}
    {%- set loop_messages = messages[1:] -%}
{%- else -%}
    {%- set first_user_prefix = '' -%}
    {%- set loop_messages = messages -%}
{%- endif -%}
{%- for message in loop_messages -%}
    {%- if (message['role'] == 'user') != (loop.index0 % 2 == 0) -%}
        {{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}
    {%- endif -%}
    {%- if message['role'] == 'assistant' -%}
        {{ '<start_of_turn>model\n' }}
        {%- generation -%}
            {{ message['content'] | trim ~ '<end_of_turn>\n' }}
        {%- endgeneration -%}
    {%- else -%}
        {{ '<start_of_turn>' ~ message['role'] ~ '\n' ~ (first_user_prefix if loop.first else '') ~ (message['content'] | trim) ~ '<end_of_turn>\n' }}
    {%- endif -%}
{%- endfor -%}
{%- if add_generation_prompt -%}
    {{ '<start_of_turn>model\n' }}
{%- endif -%}"""

ORIGINAL_CHAT_TEMPLATE = tokenizer.chat_template
if not re.search(r"\{%-?\s*generation\s*-?%\}", ORIGINAL_CHAT_TEMPLATE or ""):
    tokenizer.chat_template = GEMMA3_TRAINING_TEMPLATE

BATCH_SIZE = 2
GRAD_ACCUM = 4
EPOCHS = 3
# warmup_steps must be an int on older transformers (a 0.05 ratio is only accepted by newer ones): 5% of all optimizer steps
steps_per_epoch = math.ceil(len(train_dataset) / (BATCH_SIZE * GRAD_ACCUM))
WARMUP_STEPS = max(1, math.ceil(0.05 * steps_per_epoch * EPOCHS))

# TRL v0.14+ natively supports assistant_only_loss in SFTConfig for conversational datasets.
# In TRL >= 0.20, DataCollatorForCompletionOnlyLM has been deprecated and removed.
try:
    training_args = SFTConfig(
        output_dir="./gemma3_sinhala_output",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        logging_steps=10,
        num_train_epochs=EPOCHS,
        fp16=False,
        bf16=USE_BF16,  # set in cell 4; False on T4 (fp32 compute)
        save_strategy="epoch",
        eval_strategy="epoch",
        max_length=512,
        assistant_only_loss=True,  # Response-only loss: computes loss ONLY on assistant turns
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        args=training_args,
    )
except TypeError as e:
    # Fallback for legacy TRL versions (< 0.14) that required DataCollatorForCompletionOnlyLM
    try:
        from trl import DataCollatorForCompletionOnlyLM
    except ImportError:
        raise e from None  # not a legacy TRL: show the real error instead of the missing-collator one

    def format_prompts(example):
        return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

    train_dataset = dataset["train"].map(format_prompts, remove_columns=dataset["train"].column_names)
    val_dataset = dataset["val"].map(format_prompts, remove_columns=dataset["val"].column_names)

    collator = DataCollatorForCompletionOnlyLM(
        response_template="<start_of_turn>model\n",
        instruction_template="<start_of_turn>user\n",
        tokenizer=tokenizer,
    )

    training_args = SFTConfig(
        output_dir="./gemma3_sinhala_output",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        logging_steps=10,
        num_train_epochs=EPOCHS,
        fp16=False,
        bf16=USE_BF16,
        save_strategy="epoch",
        eval_strategy="epoch",
        dataset_text_field="text",
        max_length=512,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collator,
        processing_class=tokenizer,
        args=training_args,
    )
finally:
    # The dataset is tokenized by now; keep the saved tokenizer (Cells 7-8) on Gemma's stock template
    tokenizer.chat_template = ORIGINAL_CHAT_TEMPLATE

trainer.train()

Tokenizing train dataset:   0%|          | 0/106 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/106 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/11 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/11 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,5.218300,3.496198,3.202366,20377.000000,0.266302
2,3.429400,3.054850,2.850767,40754.000000,0.349980
3,2.709700,3.001668,2.634867,61131.000000,0.358418


TrainOutput(global_step=42, training_loss=3.546979960941133, metrics={'train_runtime': 90.2461, 'train_samples_per_second': 3.524, 'train_steps_per_second': 0.465, 'total_flos': 299320726740480.0, 'train_loss': 3.546979960941133, 'epoch': 3.0})

In [19]:
# 6. Test Interactive Multi-Persona Inference
test_messages = [
    {
        "role": "system",
        "content": "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: BEST_FRIEND]\nYou are the user's closest best friend. Responses must strictly be 1-2 spoken colloquial sentences with Sri Lankan fillers."
    },
    {"role": "user", "content": "Mata ada mara kammali ban, monada karanna hoda?"}
]

prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.7, top_p=0.9, do_sample=True)

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n🤖 SINMATE RESPONSE:", response)



🤖 SINMATE RESPONSE: ඔබේ දරුවෝ, මගේ දරුවෝ, මගේ අලුත් කෙනෙක් කියලා මගේ පටන් ගන්නෙ මට. මේ මොනක උඹ ගොඩක් ගියා උඹ ආදරෙන්?


In [20]:
# 7. Save LoRA Adapters
model.save_pretrained("sinmate-gemma-3-1b-lora")
tokenizer.save_pretrained("sinmate-gemma-3-1b-lora")
print("Adapters saved successfully to sinmate-gemma-3-1b-lora!")

Adapters saved successfully to sinmate-gemma-3-1b-lora!


In [21]:
# 8. Merge LoRA with Base Model for GGUF Export
import sys
import gc
import json
import torch

# Fix Colab incompatibility where an outdated torchao (< 0.16.0) is pre-installed.
# Bypasses torchao dispatcher check in PEFT so standard PyTorch linear LoRA merge succeeds.
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
    if hasattr(peft.import_utils, "TORCHAO_MINIMUM_VERSION"):
        peft.import_utils.TORCHAO_MINIMUM_VERSION = "0.0.0"
    for mod_name, mod in list(sys.modules.items()):
        if mod_name.startswith("peft") and hasattr(mod, "is_torchao_available"):
            setattr(mod, "is_torchao_available", lambda: False)
except Exception:
    pass

from peft import PeftModel
from transformers import AutoModelForCausalLM

# Clear CUDA cache before merging on CPU to avoid memory pressure
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Loading base model in FP16 to merge LoRA...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="cpu",
)
merged_model = PeftModel.from_pretrained(base_model, "sinmate-gemma-3-1b-lora")
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained("sinmate-gemma-3-1b-merged")
tokenizer.save_pretrained("sinmate-gemma-3-1b-merged")

# Align vocab_size in config.json with actual tokenizer vocabulary to satisfy llama.cpp GGUF converter
config_path = "sinmate-gemma-3-1b-merged/config.json"
try:
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)
    max_tok_id = max(tokenizer.get_vocab().values())
    if cfg.get("vocab_size", 0) <= max_tok_id:
        cfg["vocab_size"] = max_tok_id + 1
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)
        print(f"Aligned config.json vocab_size -> {cfg['vocab_size']} (max token ID: {max_tok_id})")
except Exception as e:
    print(f"Note: vocab_size alignment: {e}")

print("Merged model saved successfully to ./sinmate-gemma-3-1b-merged!")


Loading base model in FP16 to merge LoRA...
Aligned config.json vocab_size -> 262145 (max token ID: 262144)
Merged model saved successfully to ./sinmate-gemma-3-1b-merged!


In [22]:
# 9. Convert Merged Model to 4-bit GGUF via llama.cpp
import os
import sys
import json
import shutil
import subprocess
import tarfile
import urllib.request
import zipfile
from platform import machine, system

def run(cmd, check=False):
    """Run a command and stream its output into the cell (a bare subprocess would print to the Jupyter server's terminal)."""
    print("$", " ".join(str(c) for c in cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {' '.join(str(c) for c in cmd)}")
    return proc.returncode

# Helper to locate directories regardless of the notebook's working directory
def resolve_path(target_name):
    candidates = [
        target_name,
        os.path.join("Sinmate", target_name),
        os.path.join("..", target_name),
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return None

# 1. Locate merged model directory
merged_model_dir = resolve_path("sinmate-gemma-3-1b-merged")
if not merged_model_dir:
    raise FileNotFoundError("Could not find 'sinmate-gemma-3-1b-merged'. Please ensure Cell 8 completed successfully.")
print(f"Found merged model directory: {merged_model_dir}")

# 2. Clone llama.cpp and install conversion requirements (avoiding legacy protobuf downgrade)
llama_cpp_dir = resolve_path("llama.cpp")
if not llama_cpp_dir:
    run(["git", "clone", "https://github.com/ggml-org/llama.cpp.git"], check=True)
    llama_cpp_dir = resolve_path("llama.cpp") or os.path.abspath("llama.cpp")
print(f"Found llama.cpp directory: {llama_cpp_dir}")

# Install only what the converter needs. llama.cpp's own requirements files can pull in a different (CPU-only)
# torch build, which would silently replace this environment's CUDA torch. No --upgrade: it would force the newest
# protobuf (7.x), which breaks packages that pin protobuf < 6 (grpcio-status, google-ai-generativelanguage, ...).
pip_install = [sys.executable, "-m", "pip", "install", "-q"]
run(pip_install + [os.path.join(llama_cpp_dir, "gguf-py"), "sentencepiece", "protobuf>=5.0.0"], check=True)

# 3. Gemma 3 must be converted from its SentencePiece `tokenizer.model`. Without it llama.cpp falls back to the
#    GPT-2/BPE route, mis-detects the pre-tokenizer and writes 262145 tokens (incl. <image_soft_token>) against
#    262144 embedding rows - a GGUF that Ollama cannot load (HTTP 500 on every chat request).
tok_model_path = os.path.join(merged_model_dir, "tokenizer.model")
if not os.path.exists(tok_model_path):
    try:
        from huggingface_hub import hf_hub_download
        hf_hub_download("google/gemma-3-1b-it", "tokenizer.model", local_dir=merged_model_dir)
    except Exception as e:
        print(f"⚠️ Could not fetch tokenizer.model from google/gemma-3-1b-it: {e}")
has_spm = os.path.exists(tok_model_path)
print(f"SentencePiece tokenizer.model present: {has_spm}")

# 4. config.json vocab_size must equal the embedding rows of the weights. Earlier runs bumped it to 262145 (the
#    tokenizer's extra <image_soft_token>) and that value persists on disk: the converter then pads the vocab to
#    262145 tokens against 262144 embedding rows. Reset it from the actual weights.
if has_spm:
    import glob
    from safetensors import safe_open
    config_path = os.path.join(merged_model_dir, "config.json")
    embd_rows = None
    for st_file in sorted(glob.glob(os.path.join(merged_model_dir, "*.safetensors"))):
        with safe_open(st_file, framework="pt") as f:
            for key in f.keys():
                if key.endswith("embed_tokens.weight"):
                    embd_rows = f.get_slice(key).get_shape()[0]
                    break
        if embd_rows:
            break
    if embd_rows and os.path.exists(config_path):
        with open(config_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        if cfg.get("vocab_size") != embd_rows:
            print(f"Reset config.json vocab_size {cfg.get('vocab_size')} -> {embd_rows} (embedding rows)")
            cfg["vocab_size"] = embd_rows
            with open(config_path, "w", encoding="utf-8") as f:
                json.dump(cfg, f, indent=2)
        else:
            print(f"config.json vocab_size already matches embedding rows ({embd_rows})")

# 5. Only without tokenizer.model: fall back to the vocab_size / assertion workarounds for the BPE route
if not has_spm:
    config_path = os.path.join(merged_model_dir, "config.json")
    if os.path.exists(config_path):
        with open(config_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(merged_model_dir)
        max_tok_id = max(tok.get_vocab().values())
        if cfg.get("vocab_size", 0) <= max_tok_id:
            cfg["vocab_size"] = max_tok_id + 1
            with open(config_path, "w", encoding="utf-8") as f:
                json.dump(cfg, f, indent=2)
            print(f"Aligned config.json vocab_size -> {cfg['vocab_size']} (max token ID: {max_tok_id})")
    # Patch conversion/base.py in llama.cpp to prevent assertion crashes
    base_py = os.path.join(llama_cpp_dir, "conversion", "base.py")
    if os.path.exists(base_py):
        with open(base_py, "r", encoding="utf-8") as f:
            content = f.read()
        if "assert max(tokenizer.vocab.values()) < vocab_size" in content:
            content = content.replace(
                "assert max(tokenizer.vocab.values()) < vocab_size",
                "vocab_size = max(vocab_size, max(tokenizer.vocab.values()) + 1)"
            )
            with open(base_py, "w", encoding="utf-8") as f:
                f.write(content)
            print("Patched llama.cpp conversion/base.py assertion check")

# 6. Convert HuggingFace checkpoint to FP16 GGUF
f16_gguf = os.path.abspath("sinmate-gemma-3-1b-f16.gguf")
q4_gguf = os.path.abspath("sinmate-gemma-3-1b-q4_k_m.gguf")
# Remove GGUFs from earlier runs so a failed conversion can never leave a stale file in use
for stale in (f16_gguf, q4_gguf):
    if os.path.exists(stale):
        os.remove(stale)
convert_script = os.path.join(llama_cpp_dir, "convert_hf_to_gguf.py")
run([sys.executable, convert_script, merged_model_dir, "--outfile", f16_gguf, "--outtype", "f16"])

# 7. Locate llama-quantize: existing binary -> official prebuilt release (no compiler or root needed) -> build from source
exe = ".exe" if os.name == "nt" else ""
prebuilt_dir = os.path.join(llama_cpp_dir, "prebuilt")

def quantize_runs(binary):
    """True if the binary starts here (a prebuilt one can fail on an old glibc / missing shared library)."""
    try:
        out = subprocess.run([binary, "--help"], capture_output=True, text=True, timeout=60)
    except (OSError, subprocess.SubprocessError):
        return False
    return "usage" in (out.stdout + out.stderr).lower()

def find_quantize():
    candidates = [
        os.path.join(llama_cpp_dir, "build", "bin", "Release", "llama-quantize" + exe),  # MSVC multi-config layout
        os.path.join(llama_cpp_dir, "build", "bin", "llama-quantize" + exe),
        os.path.join(llama_cpp_dir, "build", "llama-quantize" + exe),
        os.path.join(llama_cpp_dir, "llama-quantize" + exe),
    ]
    candidates += [os.path.join(root, "llama-quantize" + exe) for root, _, files in os.walk(prebuilt_dir) if "llama-quantize" + exe in files]
    for bin_candidate in candidates:
        if os.path.exists(bin_candidate) and quantize_runs(bin_candidate):
            return os.path.abspath(bin_candidate)
    return None

def prebuilt_asset_suffix():
    """Name suffix of llama.cpp's CPU-only release archive for this OS/CPU (quantizing needs no GPU)."""
    arm = machine().lower() in ("arm64", "aarch64")
    x64 = machine().lower() in ("x86_64", "amd64")
    if system() == "Linux" and (x64 or arm):
        return "bin-ubuntu-x64.tar.gz" if x64 else "bin-ubuntu-arm64.tar.gz"
    if system() == "Darwin":
        return "bin-macos-arm64.tar.gz" if arm else "bin-macos-x64.tar.gz"
    if system() == "Windows" and (x64 or arm):
        return "bin-win-cpu-x64.zip" if x64 else "bin-win-cpu-arm64.zip"
    return None

def download_prebuilt():
    """Download and unpack the newest official llama.cpp build that ships this platform's CPU archive."""
    suffix = prebuilt_asset_suffix()
    if suffix is None:
        print(f"No prebuilt llama.cpp binary for {system()} {machine()}")
        return
    req = urllib.request.Request(
        "https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=10",
        headers={"Accept": "application/vnd.github+json", "User-Agent": "sinmate-notebook"},
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        releases = json.load(resp)
    asset = next((a for rel in releases for a in rel["assets"] if a["name"] == f"llama-{rel['tag_name']}-{suffix}"), None)
    if asset is None:
        print(f"No release archive ending in '{suffix}' among the latest llama.cpp releases")
        return
    os.makedirs(prebuilt_dir, exist_ok=True)
    archive = os.path.join(prebuilt_dir, asset["name"])
    print(f"Downloading prebuilt llama-quantize: {asset['name']} ({asset['size'] / 1e6:.0f} MB)")
    urllib.request.urlretrieve(asset["browser_download_url"], archive)
    if archive.endswith(".zip"):
        with zipfile.ZipFile(archive) as z:
            z.extractall(prebuilt_dir)
    else:
        with tarfile.open(archive) as t:
            try:
                t.extractall(prebuilt_dir, filter="data")
            except TypeError:  # Python without extraction filters
                t.extractall(prebuilt_dir)
    os.remove(archive)

quantize_bin = find_quantize()

if not quantize_bin:
    try:
        download_prebuilt()
    except Exception as e:
        print(f"⚠️ Could not fetch a prebuilt llama-quantize ({type(e).__name__}: {e})")
    quantize_bin = find_quantize()
    if not quantize_bin:
        print("⚠️ No usable prebuilt llama-quantize; building from source instead")

if not quantize_bin:
    if shutil.which("cmake") is None:
        raise RuntimeError(
            "llama-quantize is unavailable: no prebuilt binary could be downloaded/run here and cmake was not found to build one. "
            "Either install cmake and a C++ compiler (e.g. `sudo apt install cmake build-essential`, `brew install cmake`, or Visual "
            "Studio Build Tools + cmake), or download llama-quantize from https://github.com/ggml-org/llama.cpp/releases, put it "
            f"under {prebuilt_dir}, and re-run this cell."
        )
    build_dir = os.path.join(llama_cpp_dir, "build")
    run(["cmake", "-B", build_dir, llama_cpp_dir, "-DLLAMA_CURL=OFF", "-DLLAMA_BUILD_SERVER=OFF", "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF"], check=True)
    run(["cmake", "--build", build_dir, "--config", "Release", "-j", "2", "--target", "llama-quantize"], check=True)
    quantize_bin = find_quantize()
    if not quantize_bin:
        raise RuntimeError("llama-quantize was built but its binary could not be found under " + build_dir)

print(f"Using llama-quantize binary: {quantize_bin}")

# 8. Quantize to Q4_K_M (4-bit)
if os.path.exists(f16_gguf):
    if os.name != "nt":
        os.chmod(quantize_bin, 0o755)
    run([quantize_bin, f16_gguf, q4_gguf, "Q4_K_M"])
else:
    print(f"⚠️ {f16_gguf} was not found. Please review convert_hf_to_gguf logs above.")

if os.path.exists(q4_gguf):
    size_mb = os.path.getsize(q4_gguf) / (1024 * 1024)
    print(f"\n🎉 GGUF quantization complete: {q4_gguf} ({size_mb:.1f} MB)")
else:
    print("\n⚠️ Quantization output file not found. Please check logs above.")


Found merged model directory: /home/jovyan/Sinmate/sinmate-gemma-3-1b-merged
$ git clone https://github.com/ggml-org/llama.cpp.git
Cloning into 'llama.cpp'...
Found llama.cpp directory: /home/jovyan/Sinmate/llama.cpp
$ /opt/conda/bin/python -m pip install -q /home/jovyan/Sinmate/llama.cpp/gguf-py sentencepiece protobuf>=5.0.0
SentencePiece tokenizer.model present: True
Reset config.json vocab_size 262145 -> 262144 (embedding rows)
$ /opt/conda/bin/python /home/jovyan/Sinmate/llama.cpp/convert_hf_to_gguf.py /home/jovyan/Sinmate/sinmate-gemma-3-1b-merged --outfile /home/jovyan/Sinmate/sinmate-gemma-3-1b-f16.gguf --outtype f16
INFO:hf-to-gguf:Loading model: sinmate-gemma-3-1b-merged
INFO:hf-to-gguf:Model architecture: Gemma3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,                 torch.float16 --> F16, shape = {1152, 26

In [23]:
# 10. Generate Modelfile for Ollama
import os

def resolve_path(target_name):
    candidates = [
        target_name,
        os.path.join("Sinmate", target_name),
        os.path.join("..", target_name),
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return None

# Locate the quantized GGUF model produced by Cell 9
gguf_path = resolve_path("sinmate-gemma-3-1b-q4_k_m.gguf") or os.path.abspath("sinmate-gemma-3-1b-q4_k_m.gguf")
print(f"Target GGUF file: {gguf_path}")

# Gemma has no system role: the HF chat template used for fine-tuning folds the system prompt
# into the first user turn, so the Ollama TEMPLATE must do the same to match training.
modelfile_template = """FROM {GGUF_PATH}
TEMPLATE \"\"\"<start_of_turn>user
{{ if .System }}{{ .System }}

{{ end }}{{ .Prompt }}<end_of_turn>
<start_of_turn>model
{{ .Response }}<end_of_turn>
\"\"\"
PARAMETER stop "<start_of_turn>"
PARAMETER stop "<end_of_turn>"
PARAMETER temperature 0.7
PARAMETER top_p 0.9
"""

modelfile = modelfile_template.replace("{GGUF_PATH}", gguf_path)

with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)

print("\n✅ Modelfile successfully generated at:", os.path.abspath("Modelfile"))
print(f"Points to GGUF: {gguf_path}")
print("\n👉 Next Step: Run Cell 11 to install Ollama, start the daemon, and build 'sinmate-gemma3:1b' on this machine!")

Target GGUF file: /home/jovyan/Sinmate/sinmate-gemma-3-1b-q4_k_m.gguf

✅ Modelfile successfully generated at: /home/jovyan/Sinmate/Modelfile
Points to GGUF: /home/jovyan/Sinmate/sinmate-gemma-3-1b-q4_k_m.gguf

👉 Next Step: Run Cell 11 to install Ollama, start the daemon, and build 'sinmate-gemma3:1b' on this machine!


In [25]:
# 11. Setup Ollama & Build Sinmate Model on This Machine
import os
import sys
import gc
import json
import time
import shutil
import subprocess
import tarfile
import urllib.error
import urllib.request
from pathlib import Path
from platform import machine

import torch

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL_TAG = "sinmate-gemma3:1b"

def run(cmd, check=False):
    """Run a command and stream its output into the cell (a bare subprocess would print to the Jupyter server's terminal)."""
    print("$", " ".join(str(c) for c in cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {' '.join(str(c) for c in cmd)}")
    return proc.returncode

def resolve_path(target_name):
    candidates = [
        target_name,
        os.path.join("Sinmate", target_name),
        os.path.join("..", target_name),
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return None

def ollama_up():
    try:
        with urllib.request.urlopen(OLLAMA_URL + "/", timeout=2) as resp:
            return resp.status == 200
    except Exception:
        return False

def ollama_error(e, log_lines=25):
    """Readable reason for a failed Ollama request: the response body (urllib hides it) + the server log tail."""
    body = ""
    if isinstance(e, urllib.error.HTTPError):
        try:
            body = e.read().decode("utf-8", "replace")
        except Exception:
            pass
    tail = ""
    if os.path.exists("ollama_server.log"):
        with open("ollama_server.log", "r", errors="replace") as f:
            tail = "".join(f.readlines()[-log_lines:])
    return f"{e}\n{body}\n--- ollama_server.log (tail) ---\n{tail}"

def check_gguf_vocab(path):
    """Fail fast if the GGUF has more tokens than embedding rows - Ollama then answers every chat with HTTP 500."""
    try:
        from gguf import GGUFReader
        reader = GGUFReader(path)
        n_tokens = len(reader.fields["tokenizer.ggml.tokens"].data)
        embd_rows = next(int(t.shape[1]) for t in reader.tensors if t.name == "token_embd.weight")
    except (ImportError, KeyError, StopIteration) as e:
        print(f"ℹ️ Skipping GGUF vocab check ({type(e).__name__}: {e})")
        return
    print(f"GGUF vocab check: {n_tokens} tokens vs {embd_rows} embedding rows")
    if n_tokens != embd_rows:
        raise RuntimeError(
            f"❌ GGUF is broken: {n_tokens} tokens but {embd_rows} embedding rows. "
            "Re-run Cell 9 (it now converts with Gemma's SentencePiece tokenizer.model), then Cell 10 and this cell."
        )

# 1. Clean up unused PyTorch tensors to leave maximum free VRAM for Ollama
for var_name in ["model", "trainer", "base_model", "merged_model"]:
    if var_name in globals():
        del globals()[var_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Locate Ollama. On Linux it is installed from the official archive into ~/.sinmate/ollama: no root, apt or systemd
#    needed (the system installer requires sudo). macOS/Windows need the desktop installer from https://ollama.com/download.
OLLAMA_HOME = Path.home() / ".sinmate" / "ollama"  # outside the repo: Cell 3's fresh clone must not delete this
INSTALL_DONE = OLLAMA_HOME / ".install_complete"   # written last, so an interrupted download is never mistaken for an install

class CountingReader:
    """Wraps a download stream and prints progress while it is being unpacked."""
    def __init__(self, stream, total, every=250e6):
        self.stream, self.total, self.every, self.done, self.next_report = stream, total, every, 0, every
    def read(self, size=-1):
        data = self.stream.read(size)
        self.done += len(data)
        if self.done >= self.next_report:
            print(f"   ...{self.done / 1e6:.0f}" + (f" / {self.total / 1e6:.0f}" if self.total else "") + " MB downloaded")
            self.next_report += self.every
        return data

def local_ollama():
    binary = OLLAMA_HOME / "bin" / "ollama"
    return str(binary) if binary.exists() and INSTALL_DONE.exists() else None

def install_ollama_user_space(arch):
    """Stream the official Linux .tar.zst (~1.5 GB, includes the CUDA runtime libs) straight into OLLAMA_HOME."""
    run([sys.executable, "-m", "pip", "install", "-q", "zstandard"], check=True)  # Python < 3.14 cannot read .zst itself
    import zstandard
    url = f"https://ollama.com/download/ollama-linux-{arch}.tar.zst"
    print(f"📦 Downloading and unpacking Ollama into {OLLAMA_HOME} (large download, may take a few minutes)...")
    shutil.rmtree(OLLAMA_HOME, ignore_errors=True)  # drop any half-finished or older install
    OLLAMA_HOME.mkdir(parents=True)
    request = urllib.request.Request(url, headers={"User-Agent": "sinmate-notebook"})
    with urllib.request.urlopen(request, timeout=120) as resp:
        total = int(resp.headers.get("Content-Length") or 0)
        stream = zstandard.ZstdDecompressor().stream_reader(CountingReader(resp, total))
        with tarfile.open(fileobj=stream, mode="r|") as archive:
            try:
                archive.extractall(OLLAMA_HOME, filter="data")
            except TypeError:  # Python without extraction filters
                archive.extractall(OLLAMA_HOME)
    INSTALL_DONE.touch()

OLLAMA_BIN = shutil.which("ollama") or local_ollama()
if OLLAMA_BIN is None:
    linux_arch = {"x86_64": "amd64", "amd64": "amd64", "aarch64": "arm64", "arm64": "arm64"}.get(machine().lower())
    if sys.platform.startswith("linux") and linux_arch:
        try:
            install_ollama_user_space(linux_arch)
        except Exception as e:
            raise RuntimeError(
                f"❌ Could not install Ollama automatically ({type(e).__name__}: {e}). Install it manually from "
                "https://ollama.com/download, then re-run this cell, or skip to Cell 12 (HuggingFace fallback)."
            ) from e
        OLLAMA_BIN = local_ollama()
    if OLLAMA_BIN is None:
        raise RuntimeError(
            "❌ Ollama not found. Install it from https://ollama.com/download, restart the Jupyter server if `ollama` is still "
            "not on PATH, then re-run this cell. To chat without Ollama, skip to Cell 12 (HuggingFace fallback)."
        )
print(f"✅ Using Ollama: {OLLAMA_BIN}")
run([OLLAMA_BIN, "--version"])

# 3. Start background Ollama daemon if not running
if ollama_up():
    print("✅ Ollama server is already running.")
else:
    print("🚀 Starting Ollama daemon in background...")
    log_fp = open("ollama_server.log", "a")
    detach = (
        {"creationflags": subprocess.DETACHED_PROCESS | subprocess.CREATE_NEW_PROCESS_GROUP}
        if os.name == "nt"
        else {"start_new_session": True}
    )
    subprocess.Popen([OLLAMA_BIN, "serve"], stdout=log_fp, stderr=subprocess.STDOUT, **detach)
    for attempt in range(30):
        if ollama_up():
            print(f"✅ Ollama server is live at {OLLAMA_URL} (attempt {attempt + 1})")
            break
        time.sleep(1)
    else:
        with open("ollama_server.log", "r") as f:
            print("".join(f.readlines()[-20:]))
        raise RuntimeError("❌ Ollama server did not respond - see log output above.")

# 4. Locate Modelfile and GGUF
modelfile_path = resolve_path("Modelfile")
gguf_path = resolve_path("sinmate-gemma-3-1b-q4_k_m.gguf")

if not modelfile_path:
    raise FileNotFoundError("❌ 'Modelfile' not found. Please run Cell 10 first!")
if not gguf_path:
    raise FileNotFoundError("❌ 'sinmate-gemma-3-1b-q4_k_m.gguf' not found. Please run Cell 9 first!")

print(f"Modelfile: {modelfile_path}")
print(f"GGUF: {gguf_path}")
check_gguf_vocab(gguf_path)

# Ensure Modelfile points to the exact absolute GGUF path
with open(modelfile_path, "r", encoding="utf-8") as f:
    mf_lines = f.read().splitlines()

mf_lines = [f"FROM {gguf_path}" if line.startswith("FROM ") else line for line in mf_lines]

with open(modelfile_path, "w", encoding="utf-8") as f:
    f.write("\n".join(mf_lines) + "\n")

# 5. Build the Sinmate model in Ollama
print(f"\n🔨 Building Ollama model '{MODEL_TAG}' from Modelfile...")
run([OLLAMA_BIN, "create", MODEL_TAG, "-f", modelfile_path], check=True)

# 6. List Ollama models
print("\n📋 Available Ollama models:")
run([OLLAMA_BIN, "list"])

# 7. Quick verification through the HTTP API (the `ollama run` CLI prints spinner escape codes into the cell output)
print("\n🧪 Running quick verification prompt via Ollama API...")
verify_payload = json.dumps({
    "model": MODEL_TAG,
    "stream": False,
    "messages": [
        {"role": "system", "content": "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: BEST_FRIEND]\nYou are the user's closest best friend. Responses must strictly be 1-2 spoken colloquial sentences with Sri Lankan fillers."},
        {"role": "user", "content": "Mata ada mara kammali ban, monada karanna hoda?"},
    ],
    "options": {"temperature": 0.7, "top_p": 0.9, "num_predict": 64},
}).encode("utf-8")
verify_req = urllib.request.Request(OLLAMA_URL + "/api/chat", data=verify_payload, headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(verify_req, timeout=300) as resp:
        print("🤖 SINMATE RESPONSE:", json.loads(resp.read().decode("utf-8"))["message"]["content"])
except Exception as e:
    print(f"⚠️ Verification request failed: {ollama_error(e)}")

$ /opt/conda/bin/python -m pip install -q zstandard
📦 Downloading and unpacking Ollama into /home/jovyan/.sinmate/ollama (large download, may take a few minutes)...
   ...250 / 1509 MB downloaded
   ...500 / 1509 MB downloaded
   ...750 / 1509 MB downloaded
   ...1000 / 1509 MB downloaded
   ...1250 / 1509 MB downloaded
   ...1500 / 1509 MB downloaded
✅ Using Ollama: /home/jovyan/.sinmate/ollama/bin/ollama
$ /home/jovyan/.sinmate/ollama/bin/ollama --version
🚀 Starting Ollama daemon in background...
✅ Ollama server is live at http://127.0.0.1:11434 (attempt 2)
Modelfile: /home/jovyan/Sinmate/Modelfile
GGUF: /home/jovyan/Sinmate/sinmate-gemma-3-1b-q4_k_m.gguf
GGUF vocab check: 262144 tokens vs 262144 embedding rows

🔨 Building Ollama model 'sinmate-gemma3:1b' from Modelfile...
$ /home/jovyan/.sinmate/ollama/bin/ollama create sinmate-gemma3:1b -f /home/jovyan/Sinmate/Modelfile
gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components

In [ ]:
# 12. Real-Time Interactive Multi-Persona Chat
import os
import sys
import json
import urllib.error
import urllib.request

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL_TAG = "sinmate-gemma3:1b"
MAX_HISTORY = 6  # last N messages kept as conversation context

# 1. Define Persona Prompts (Kathitha Bhashawa / Sri Lankan Spoken Sinhala)
PERSONAS = {
    "1": ("BEST_FRIEND", "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: BEST_FRIEND]\nYou are the user's closest best friend. Responses must strictly be 1-2 spoken colloquial sentences with Sri Lankan fillers."),
    "2": ("ROMANTIC_PARTNER", "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: ROMANTIC_PARTNER]\nYou are the user's loving, affectionate romantic partner. Responses must strictly be 1-2 warm, spoken colloquial sentences."),
    "3": ("COUNSELLOR", "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: COUNSELLOR]\nYou are a calm, deeply empathetic listener. Responses must strictly be 1-2 grounded, supportive spoken colloquial sentences."),
    "4": ("PLAYFUL_SIBLING", "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: PLAYFUL_SIBLING]\nYou are the user's witty, teasing sibling. Responses must strictly be 1-2 playful, humorous spoken colloquial sentences."),
}

# 2. Check if the Ollama engine is up and has the Sinmate model
def is_ollama_ready():
    try:
        with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=2) as r:
            data = json.loads(r.read().decode("utf-8"))
            return any("sinmate-gemma3" in m.get("name", "") for m in data.get("models", []))
    except Exception:
        return False

ollama_available = is_ollama_ready()

def ollama_error(e, log_lines=25):
    """Readable reason for a failed Ollama request: the response body (urllib hides it) + the server log tail."""
    body = ""
    if isinstance(e, urllib.error.HTTPError):
        try:
            body = e.read().decode("utf-8", "replace")
        except Exception:
            pass
    tail = ""
    if os.path.exists("ollama_server.log"):
        with open("ollama_server.log", "r", errors="replace") as f:
            tail = "".join(f.readlines()[-log_lines:])
    return f"{e}\n{body}\n--- ollama_server.log (tail) ---\n{tail}"

def build_messages(user_text, persona_key, history):
    _, system_prompt = PERSONAS.get(persona_key, PERSONAS["1"])
    messages = [{"role": "system", "content": system_prompt}]
    if history:
        messages.extend(history[-MAX_HISTORY:])
    messages.append({"role": "user", "content": user_text})
    return messages

# 3. Streaming chat functions (both return the full reply so it can be kept as history)
def chat_via_ollama(user_text, persona_key="1", history=None):
    persona_name = PERSONAS.get(persona_key, PERSONAS["1"])[0]
    payload = json.dumps({
        "model": MODEL_TAG,
        "messages": build_messages(user_text, persona_key, history),
        "stream": True,
        "options": {
            "temperature": 0.7,
            "top_p": 0.9,
            "num_predict": 128,
            "stop": ["<start_of_turn>", "<end_of_turn>"],
        },
    }).encode("utf-8")

    req = urllib.request.Request(
        OLLAMA_URL + "/api/chat",
        data=payload,
        headers={"Content-Type": "application/json"},
    )

    print(f"\n[{persona_name}] Sinmate: ", end="")
    full_response = []
    with urllib.request.urlopen(req, timeout=300) as resp:
        for line in resp:
            if not line.strip():
                continue
            token = json.loads(line.decode("utf-8")).get("message", {}).get("content", "")
            if token:
                sys.stdout.write(token)
                sys.stdout.flush()
                full_response.append(token)
    print()
    return "".join(full_response)

# Hugging Face fallback if Ollama is not running
hf_model = None
hf_tokenizer = None

def init_hf_fallback():
    global hf_model, hf_tokenizer
    if hf_model is not None:
        return

    os.environ["TRANSFORMERS_NO_TF"] = "1"
    os.environ["USE_TF"] = "0"
    try:
        import transformers.utils.import_utils as tr_import
        tr_import.is_tf_available = lambda: False
        tr_import.is_flax_available = lambda: False
        tr_import.is_torch_available = lambda: True
    except Exception:
        pass

    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    model_dirs = [
        "sinmate-gemma-3-1b-merged",
        "Sinmate/sinmate-gemma-3-1b-merged",
    ]
    m_path = "google/gemma-3-1b-it"
    for p in model_dirs:
        if os.path.exists(p):
            m_path = os.path.abspath(p)
            break

    # Gemma 3 overflows in fp16 (see Cell 4): bf16 on Ampere+, fp32 on the T4
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float32

    print(f"Loading HuggingFace fallback from: {m_path} ...")
    hf_tokenizer = AutoTokenizer.from_pretrained(m_path)
    if hf_tokenizer.pad_token is None:
        hf_tokenizer.pad_token = hf_tokenizer.eos_token
    hf_model = AutoModelForCausalLM.from_pretrained(m_path, dtype=dtype, device_map="auto")
    hf_model.eval()

def chat_via_hf(user_text, persona_key="1", history=None):
    init_hf_fallback()
    import torch
    from transformers import TextStreamer

    persona_name = PERSONAS.get(persona_key, PERSONAS["1"])[0]
    messages = build_messages(user_text, persona_key, history)

    prompt = hf_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = hf_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(hf_model.device)
    streamer = TextStreamer(hf_tokenizer, skip_prompt=True, skip_special_tokens=True)

    print(f"\n[{persona_name}] Sinmate: ", end="")
    with torch.no_grad():
        outputs = hf_model.generate(
            **inputs,
            streamer=streamer,
            max_new_tokens=64,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )
    return hf_tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def chat_sinmate(user_text, persona_key="1", history=None):
    global ollama_available
    if ollama_available:
        try:
            return chat_via_ollama(user_text, persona_key, history)
        except urllib.error.URLError as e:  # HTTPError is a subclass
            print(f"\n⚠️ Ollama request failed - falling back to HuggingFace Transformers:\n{ollama_error(e)}")
            ollama_available = False
    return chat_via_hf(user_text, persona_key, history)

# 4. Print status & run automated multi-persona tests
print("=" * 60)
if ollama_available:
    print(f"🚀 SINMATE REAL-TIME CHAT (ENGINE: Ollama - {MODEL_TAG})")
else:
    print("🚀 SINMATE REAL-TIME CHAT (ENGINE: HuggingFace Transformers)")
print("=" * 60)

sample_tests = [
    ("Mata ada mara kammali ban, monada karanna hoda?", "1"),
    ("Mata ada godak mahansi darling.", "2"),
    ("Mata godak bayai exam eka gana.", "3"),
    ("Mage chocolate eka kawe oyada?", "4"),
]

for prompt_text, p_key in sample_tests:
    print(f"\nUser ({PERSONAS[p_key][0]}): {prompt_text}")
    chat_sinmate(prompt_text, persona_key=p_key)

print("\n" + "=" * 60)
print("💬 Interactive Real-Time Chat:")
print("Commands: 'p 1-4' to change persona | 'exit' to quit")
print("Personas: [1] Best Friend   [2] Romantic Partner   [3] Counsellor   [4] Playful Sibling")
print("=" * 60)

try:
    current_persona = "1"
    history = []
    while True:
        user_input = input("\nYou (or 'p 1-4' to switch, 'exit' to quit): ").strip()
        if not user_input or user_input.lower() in ["exit", "quit"]:
            print("Chat session ended. Sinmate signing off!")
            break
        if user_input.startswith("p ") and user_input[2:] in PERSONAS:
            current_persona = user_input[2:]
            history = []  # new persona starts a fresh conversation
            print(f"Switched persona to: {PERSONAS[current_persona][0]}")
            continue
        reply = chat_sinmate(user_input, persona_key=current_persona, history=history)
        history += [{"role": "user", "content": user_input}, {"role": "assistant", "content": reply}]
except (EOFError, KeyboardInterrupt):
    print("\nSession finished.")

🚀 SINMATE REAL-TIME CHAT (ENGINE: Ollama - sinmate-gemma3:1b)

User (BEST_FRIEND): Mata ada mara kammali ban, monada karanna hoda?

[BEST_FRIEND] Sinmate: කැස්යක් හොඳ ආදරේ වුණා මගේ රතුරේ මට හොඳටම දැනෙන්නේ කියලා මගේ විස්මයි. දැන් හොඳටම කතා කරන්නේ කොළඹට ආදරේ මට ඉතින්ම.

User (ROMANTIC_PARTNER): Mata ada godak mahansi darling.

[ROMANTIC_PARTNER] Sinmate: ඔය හොඳයි ගෙදර ගිහින්, මගේ සතුටු එකක් ඒක නෙල්ලේ. දැන් ඒක කියන්නට බං නොවෙයි, මගේ ඇඟට හොඳ ආදරේ එකක්.

User (COUNSELLOR): Mata godak bayai exam eka gana.

[COUNSELLOR] Sinmate: ඔබේ රත්තරවම ඇඟලු සුදු වුණේට විතර මගේ රත්තරවම ඇඟලු සුදු වුණේට ඇඟලු සුදු වුණේට බේහුවෙන්න වුණේ නැතුව හෙමින් ඇඟලු සුදු වුණේට ඇඟලු සුදු වුණේට ඇඟලු සුදු වුණේට බේහුවෙන්න වුණේ නැතුව අහලා වුණේ.

User (PLAYFUL_SIBLING): Mage chocolate eka kawe oyada?

[PLAYFUL_SIBLING] Sinmate: ආරණ ආතනට ආතනට ඒක නැතුව ආකෝච ේවා